# Fase defensiva — Jardine / América

Dónde y con qué frecuencia presiona, intensidad de presión (criterio empírico por percentiles), altura de línea defensiva y velocidad de reorganización tras perder el balón.


In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import matplotlib.pyplot as plt
from framework_analitico import (
    identify_pressure_phases,
    fases_presion_con_zona,
    clasificar_intensidad_presion,
    normalizar_direccion_ataque,
    altura_linea_defensiva,
    tiempo_reaccion_perdida,
)

EQUIPO = "América"  # cambia esto para reaplicar el analisis a otro entrenador/equipo

events_jardine = pd.read_csv("../data/raw/events_jardine_america.csv", low_memory=False)
print(f"Eventos cargados: {len(events_jardine)}")

# Normalizar dirección de ataque ANTES de cualquier análisis de zona
events_norm = normalizar_direccion_ataque(events_jardine, equipo=EQUIPO)


## 1. ¿Dónde presiona? Zona de las fases de presión


In [ ]:
fases_zona = fases_presion_con_zona(events_norm)
fases_america = fases_zona[fases_zona.team == EQUIPO]

print(f"Total de fases de presión de {EQUIPO}: {len(fases_america)}")
print(fases_america.zona.value_counts(normalize=True).mul(100).round(1))

zona_counts = fases_america.zona.value_counts().reindex(["defensivo", "medio", "ofensivo"])
zona_counts.plot(kind="bar", color=["#c0392b", "#e67e22", "#27ae60"], figsize=(6, 5))
plt.title(f"Zona de las fases de presión — {EQUIPO}")
plt.ylabel("Número de fases de presión")
plt.tight_layout()
plt.savefig("../outputs/figures/zona_fases_presion.png", dpi=150, bbox_inches="tight")
plt.show()


## 2. Intensidad de presión: clasificación empírica por percentiles

Criterio defendible pedido por el organizador: no se impone un umbral fijo de duración para distinguir presión "baja/media/alta" — se calculan los percentiles 33/66 de la duración de las fases del PROPIO equipo, y se cita el benchmark de la literatura (ventana de contrapresión de ~5 segundos) como referencia de contraste.


In [ ]:
fases_presion = identify_pressure_phases(events_norm)
fases_america_presion = fases_presion[fases_presion.team == EQUIPO]

print(f"Total de fases de presión de {EQUIPO}: {len(fases_america_presion)}")
print()
print("Distribución de duración de fase (segundos):")
print(fases_america_presion.duracion_fase.value_counts().sort_index().head(15))
print()
print("Distribución de número de eventos Pressure por fase:")
print(fases_america_presion.n_eventos_presion.value_counts().sort_index().head(15))

pct_duracion_cero = (fases_america_presion.duracion_fase == 0).mean() * 100
print(f"\n% de fases con duración = 0s (un solo evento Pressure aislado): {pct_duracion_cero:.1f}%")

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

conteo_duracion = fases_america_presion.duracion_fase.value_counts().sort_index()
axs[0].bar(conteo_duracion.index, conteo_duracion.values, color="#8e44ad")
axs[0].set_title("Duración de la fase de presión (s)")
axs[0].set_xlabel("Segundos")
axs[0].set_ylabel("Número de fases")
axs[0].set_xlim(-0.5, 15)

conteo_eventos = fases_america_presion.n_eventos_presion.value_counts().sort_index()
axs[1].bar(conteo_eventos.index, conteo_eventos.values, color="#16a085")
axs[1].set_title("Número de eventos Pressure por fase")
axs[1].set_xlabel("Eventos")
axs[1].set_xlim(0.5, 15)

fig.suptitle(f"Distribución de fases de presión — {EQUIPO} (antes de elegir el corte)")
plt.tight_layout()
plt.savefig("../outputs/figures/distribucion_fases_presion.png", dpi=150, bbox_inches="tight")
plt.show()


**Nota metodológica:** si la mayoría de las fases tiene duración 0s (un solo evento Pressure aislado, sin otro evento del mismo equipo a ≤3s), los percentiles de duración colapsan y la clasificación en baja/media/alta se rompe (es justo lo que pasó la primera vez que corrimos esto). Por eso `clasificar_intensidad_presion` ahora trabaja en dos etapas: primero separa fases "puntuales" (1 solo evento → baja) de "sostenidas" (2+ eventos encadenados), y SOLO dentro de las sostenidas calcula un corte por percentil (mediana por defecto) para separar media de alta. Esto sí produce las 3 categorías de forma defendible ante el organizador.

In [ ]:
fases_clasificadas, info = clasificar_intensidad_presion(fases_presion, equipo=EQUIPO)

print(f"% de fases puntuales (1 solo evento, clasificadas directo como 'baja'): {info['pct_puntual']}%")
print(f"Corte usado entre 'media' y 'alta' dentro de las sostenidas (percentil {info['percentil_usado']} de duración): {info['corte_sostenida_segundos']}s")
print()
print("Conteo por categoría de intensidad:")
for cat, n in info["conteos"].items():
    print(f"  {cat}: {n} ({n / len(fases_clasificadas) * 100:.1f}%)")
print()
print(f"Duración promedio de fase de presión: {fases_clasificadas.duracion_fase.mean():.1f}s")
print("(referencia de literatura: ventana estándar de contrapresión/counterpress ≈ 5s)")


## 3. Altura de la línea defensiva


In [ ]:
altura = altura_linea_defensiva(events_norm, equipo=EQUIPO)

print(f"Altura de línea promedio (x, 0=propia portería, 120=portería rival): {altura.altura_linea_x.mean():.1f}")
print(f"Desviación estándar entre partidos: {altura.altura_linea_x.std():.1f}")
print(f"Partido con línea más alta: {altura.altura_linea_x.max():.1f}  |  más baja: {altura.altura_linea_x.min():.1f}")

altura.altura_linea_x.plot(kind="hist", bins=20, color="#2980b9", figsize=(7, 5))
plt.axvline(altura.altura_linea_x.mean(), color="black", linestyle="--", label="Promedio")
plt.title(f"Distribución de altura de línea defensiva por partido — {EQUIPO}")
plt.xlabel("Altura de línea (x normalizada)")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/figures/altura_linea_defensiva.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Velocidad de reacción tras perder el balón (contrapresión)

**Nota:** esta celda recorre cada pérdida de balón del equipo evento por evento, puede tardar varios minutos en correr sobre las 3 temporadas completas.


In [ ]:
reaccion = tiempo_reaccion_perdida(events_norm, equipo=EQUIPO)

pct_contrapresiona = reaccion.contrapresiono.mean() * 100
tiempo_prom = reaccion.loc[reaccion.contrapresiono, "tiempo_reaccion"].mean()

print(f"Total de pérdidas de balón analizadas: {len(reaccion)}")
print(f"% de pérdidas tras las que contrapresiona dentro de la ventana: {pct_contrapresiona:.1f}%")
print(f"Tiempo promedio de reacción (cuando contrapresiona): {tiempo_prom:.2f}s")


## 5. Síntesis del bloque defensivo

Con estos cuatro resultados (zona de presión, intensidad por percentiles, altura de línea y velocidad de contrapresión) queda cubierta la pregunta "¿cómo reacciona defensivamente el equipo?" del Plan Maestro. El siguiente paso natural es `03_ofensivo_vs_defensivo.ipynb`, que usa `perfil_ofensivo_defensivo` para comparar a América contra sus rivales dentro del mismo dataset.
